# Testing my custom logistic regression model

In [ ]:
import sys
sys.path.append('..')

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from models.logistic_regression import LogisticRegression

In [ ]:
heart_disease = fetch_ucirepo(id=45)

X_raw: pd.DataFrame = heart_disease.data.features
y_raw: pd.Series = (heart_disease.data.targets['num'] > 0).astype(int)

In [ ]:
mask = ~np.asarray(pd.isna(X_raw).any(axis=1))

X = X_raw[mask]
y = y_raw[mask]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUM_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

prep = ColumnTransformer(
    transformers=[
        ('stand', StandardScaler(), NUM_COLS),
    ],
    remainder="passthrough",
)

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y.to_numpy(), test_size=0.2, stratify=y, random_state=42
)


def run_experiment(**params):
    np.random.seed(42)

    pipe = Pipeline([
        ('prep', clone(prep)),
        ('clf', LogisticRegression(**params)),
    ])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    print(f'{params} | test accuracy: {accuracy_score(y_test, y_pred):.3f} | test f1: {f1_score(y_test, y_pred):.3f} | test precision: {precision_score(y_test, y_pred, zero_division=0)} | test recall: {recall_score(y_test, y_pred, zero_division=0)}')

    pipe.named_steps['clf'].plot_history()
    plt.show()

    return pipe

## Learning rate impact

In [ ]:
for learning_rate in [1, 0.5, 0.1, 0.01]:
    run_experiment(learning_rate=learning_rate)

## Weight initial distribution 

In [ ]:
for random_weights in [False, True]:
    run_experiment(random_weights=random_weights)

## Number of epoches

In [ ]:
for epoches in [10, 100, 1000, 5000]:
    run_experiment(epoches=epoches)

## Batch size

In [ ]:
for batch_size in [1, 16, 32, 128, len(X_train)]:
    run_experiment(batch_size=batch_size)